# Module 3: One-Hot Encoding and Bag of Words

**Objective:** Understand how to convert text (which is unstructured) into numbers (which models need). We cover two foundational approaches and their limitations, which motivate the embeddings approach in Module 4.

---

## Part A: One-Hot Encoding (OHE)

### 3.1 The Problem: Machines Need Numbers

Suppose you have a column of colors: ["red", "blue", "green"]. A model cannot process strings directly. We need a numerical representation.

**Wrong approach**: Assign integers (red=0, blue=1, green=2). This implies an ordering (green > blue > red) and distances (green is "twice" blue), which is meaningless for categorical data.

**Correct approach**: One-Hot Encoding. Each category becomes its own binary column:

| Original | red | blue | green |
|---|---|---|---|
| red | 1 | 0 | 0 |
| blue | 0 | 1 | 0 |
| green | 0 | 0 | 1 |

Properties of OHE:
- Each vector has exactly one 1 and the rest 0s.
- All vectors are **orthogonal** (perpendicular) to each other.
- All vectors have the **same distance** from each other.
- This means OHE treats all categories as **equally different** -- which is both a strength (no false ordering) and a weakness (no notion of similarity).

### 3.2 One-Hot Encoding for Categorical Features

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Sample dataset: predicting if someone will buy a laptop
data = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'blue', 'red', 'green', 'blue', 'red'],
    'size': ['S', 'M', 'L', 'S', 'L', 'M', 'L', 'S'],
    'brand': ['Apple', 'Dell', 'HP', 'Apple', 'Dell', 'HP', 'Apple', 'Dell'],
    'bought': [1, 0, 1, 0, 1, 1, 0, 1]
})

print('Original data:')
print(data)
print()

In [ ]:
# --- Method 1: pandas get_dummies (simplest) ---
encoded_pd = pd.get_dummies(data, columns=['color', 'size', 'brand'], dtype=int)
print('One-Hot Encoded with pandas:')
print(encoded_pd)
print()
print(f'Original shape: {data.shape}')
print(f'Encoded shape:  {encoded_pd.shape}')
print('Notice: 3 categorical columns with 3 categories each became 9 binary columns.')

In [ ]:
# --- Method 2: sklearn OneHotEncoder (for ML pipelines) ---
encoder = OneHotEncoder(sparse_output=False, drop=None)
categorical_cols = ['color', 'size', 'brand']
encoded_array = encoder.fit_transform(data[categorical_cols])

print('Feature names:', encoder.get_feature_names_out(categorical_cols))
print()
print('Encoded array:')
print(encoded_array)
print()
print('Key advantage of sklearn: the encoder object can transform NEW data at inference time.')

### 3.3 One-Hot Encoding for Words

OHE can also represent individual words. Given a vocabulary, each word becomes a vector with a single 1.

Limitation: if vocabulary has 10,000 words, each vector is 10,000-dimensional with 9,999 zeros. This is **extremely sparse** and captures **no meaning** -- "king" and "queen" are as different as "king" and "pizza".

In [ ]:
# One-hot encoding for words
vocabulary = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran']
word_to_idx = {word: i for i, word in enumerate(vocabulary)}

def one_hot_word(word, vocab_dict):
    vec = np.zeros(len(vocab_dict))
    if word in vocab_dict:
        vec[vocab_dict[word]] = 1
    return vec

# Encode some words
for word in ['cat', 'dog', 'mat']:
    vec = one_hot_word(word, word_to_idx)
    print(f'{word:5s} -> {vec}')

print()
# Show that all pairs are equally distant
cat_vec = one_hot_word('cat', word_to_idx)
dog_vec = one_hot_word('dog', word_to_idx)
mat_vec = one_hot_word('mat', word_to_idx)

print(f'Distance(cat, dog) = {np.linalg.norm(cat_vec - dog_vec):.4f}')
print(f'Distance(cat, mat) = {np.linalg.norm(cat_vec - mat_vec):.4f}')
print('All equal distance -- OHE captures NO semantic similarity.')

---

## Part B: Bag of Words (BoW)

### 3.4 From Words to Documents

One-hot encoding represents single words. But how do we represent **an entire document** (a sentence, paragraph, or article)?

**Bag of Words** counts how many times each vocabulary word appears in a document. The result is a vector of word counts (or frequencies).

"Bag" means we **discard word order** -- "dog bites man" and "man bites dog" get the same representation. This is a major limitation, but BoW is simple and surprisingly effective for many tasks.

### 3.5 How BoW Works

1. Build a vocabulary from all documents.
2. For each document, count occurrences of each vocabulary word.
3. The result is a matrix: rows = documents, columns = vocabulary words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Sample corpus
corpus = [
    'The cat sat on the mat',
    'The dog sat on the log',
    'The cat and the dog are friends',
    'Dogs and cats are great pets',
]

# --- Bag of Words ---
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(corpus)

print('Vocabulary:', count_vec.get_feature_names_out())
print()
print('Bag of Words matrix (rows = documents, columns = words):')
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print(bow_df)
print()
print('Notice: "the" appears 2 times in Doc 1, 2 times in Doc 2, etc.')
print('But "the" is not very informative -- it appears everywhere.')

### 3.6 TF-IDF: Smarter Than Raw Counts

**TF-IDF** (Term Frequency - Inverse Document Frequency) solves the problem of common words dominating.

- **TF**: How often a word appears in THIS document. (Same as BoW count.)
- **IDF**: log(total docs / docs containing this word). Rare words get higher IDF.
- **TF-IDF = TF x IDF**: Words that are frequent in a document BUT rare across all documents get the highest scores.

This means "the" (appears everywhere) gets a low TF-IDF, while a topic-specific word like "friends" gets a higher score.

In [ ]:
# --- TF-IDF ---
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    np.round(tfidf_matrix.toarray(), 3),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print('TF-IDF matrix:')
print(tfidf_df)
print()
print('Notice how common words like "the" have lower scores than unique words.')

### 3.7 Using BoW for a Simple Text Classification Task

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Slightly larger dataset for classification
texts = [
    'I love this movie it is great',
    'This film is wonderful and amazing',
    'Excellent movie with great acting',
    'I enjoyed this film very much',
    'Best movie I have seen this year',
    'This movie is terrible and boring',
    'I hated this film it was awful',
    'Worst movie ever very bad acting',
    'I did not enjoy this film at all',
    'Terrible movie do not watch this',
    'Great story and beautiful cinematography',
    'Horrible plot and bad dialogue',
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0]  # 1 = positive, 0 = negative

# Convert to BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out())}')
print(f'Matrix shape: {X.shape}  (12 documents, {X.shape[1]} unique words)')

# Train a simple classifier
clf = MultinomialNB()
clf.fit(X, labels)

# Test on new sentences
test_texts = ['This movie is great I love it', 'Terrible film very boring']
X_test = vectorizer.transform(test_texts)
predictions = clf.predict(X_test)

for text, pred in zip(test_texts, predictions):
    sentiment = 'Positive' if pred == 1 else 'Negative'
    print(f'  "{text}" -> {sentiment}')

### 3.8 Key Takeaways

| Method | Represents | Captures Meaning? | Captures Order? | Dimensionality |
|---|---|---|---|---|
| One-Hot Encoding | A single word/category | No | N/A | Vocabulary size (sparse) |
| Bag of Words | A document | Partially (via word presence) | No | Vocabulary size (sparse) |
| TF-IDF | A document | Better (weights important words) | No | Vocabulary size (sparse) |

All three produce **sparse, high-dimensional** representations with **no notion of semantic similarity**. This motivates Module 4, where we learn **dense, meaningful embeddings** that capture the fact that "king" is closer to "queen" than to "pizza".

---

### 3.9 Connecting the Modules

- **Module 1**: We fed **numerical features** (area) into linear regression. But what if the feature is "color" or a sentence?
- **Module 3 (this module)**: OHE and BoW convert categories/text into numbers. These become the input features for models in Module 1 and Module 2.
- **Module 4 (next)**: Embeddings are a smarter way to do this conversion, producing dense vectors that capture meaning.